#### EDA 05: Reviews table
**File:** `olist_order_reviews_dataset.csv`  |  **Goal:** understand the review scores and comments, using this table only. 

#### Load and first look
**Question:** how big is the table and what does a row look like?

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import unicodedata

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

RAW = Path("../../data/raw")                      # path from the notebook folder to the raw data
reviews_raw = pd.read_csv(RAW / "olist_order_reviews_dataset.csv")
reviews = reviews_raw.copy()                      # all changes happen on the copy, raw stays untouched

print("shape:", reviews.shape)
print(reviews.dtypes)
reviews.head(3)

shape: (99224, 7)
review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str
dtype: object


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24


- 99,224 rows and 7 columns: review id, order id, score (1 to 5), comment title, comment message, creation date and answer timestamp.
- Score is numeric. The two comment columns are free text in Brazilian Portuguese. The two date columns are still text.
- One row is one review. The first rows have no comment text, which is common (see the missing values section).

#### Date columns
**Question:** what period do the reviews cover, and what kind of date is each column?

In [2]:
DATE_COLS = ["review_creation_date", "review_answer_timestamp"]
for c in DATE_COLS:
    reviews[c] = pd.to_datetime(reviews[c])

display(reviews[DATE_COLS].agg(["min", "max"]).T)

# Is the creation date only a day (midnight), or a full timestamp?
at_midnight = reviews["review_creation_date"] == reviews["review_creation_date"].dt.normalize()
print("creation dates at midnight:", at_midnight.sum(), "of", len(reviews))

,min,max
review_creation_date,2016-10-02 00:00:00,2018-08-31 00:00:00
review_answer_timestamp,2016-10-07 18:32:28,2018-10-29 12:27:35


creation dates at midnight: 99139 of 99224


- Reviews were created from 2016-10-02 to 2018-08-31. Answers run until 2018-10-29.
- `review_creation_date` is a day (midnight) for 99,139 of 99,224 rows. `review_answer_timestamp` is a full timestamp. The creation date is most likely the day the survey was sent, and the answer timestamp the moment the customer answered.
- The last review was created on 2018-08-31, so the reviews table ends earlier than the orders table (orders run to 2018-10-17).

#### Keys and duplicates
**Question:** what identifies a review, and is any row repeated?

In [3]:
print("fully duplicated rows              :", reviews.duplicated().sum())
print("repeated (review_id, order_id)     :", reviews.duplicated(["review_id", "order_id"]).sum())
print("distinct review_id                 :", reviews["review_id"].nunique(), "| rows:", len(reviews))
print("distinct order_id                  :", reviews["order_id"].nunique())

# review_id repeated on several rows?
by_id = reviews.groupby("review_id").agg(n_rows=("order_id", "size"), n_orders=("order_id", "nunique"))
print("\nreview_ids on more than one row    :", (by_id["n_rows"] > 1).sum(),
      "| rows in them:", by_id.loc[by_id["n_rows"] > 1, "n_rows"].sum())
print("review_ids linked to >1 order      :", (by_id["n_orders"] > 1).sum())
print("maximum rows for one review_id     :", by_id["n_rows"].max())

shared = reviews[reviews.duplicated("review_id", keep=False)].sort_values("review_id")
content_cols = ["review_score", "review_comment_title", "review_comment_message", "review_creation_date", "review_answer_timestamp"]
same_content = shared.groupby("review_id")[content_cols].nunique(dropna=False).max(axis=1) == 1
print("shared review_ids with identical score, text and dates on every row:", same_content.sum(), "of", shared["review_id"].nunique())

shared[["review_id", "order_id", "review_score", "review_creation_date"]].head(4)

fully duplicated rows              : 0
repeated (review_id, order_id)     : 0
distinct review_id                 : 98410 | rows: 99224
distinct order_id                  : 98673

review_ids on more than one row    : 789 | rows in them: 1603
review_ids linked to >1 order      : 789
maximum rows for one review_id     : 3
shared review_ids with identical score, text and dates on every row: 789 of 789


,review_id,order_id,review_score,review_creation_date
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,2018-03-07
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,2018-03-07
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,2017-09-21
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,2017-09-21


- No fully duplicated rows, and the pair (`review_id`, `order_id`) is unique. This pair is the key of the table.
- `review_id` alone is not a key: 789 review ids appear on 1,603 rows. Each of them is linked to more than one order.
- All 789 shared review ids have the same score, text and dates on every row (count above). It looks like one review copied to several orders.
- **Decision:** keep the pair as the key. The duplicates are not removed, because each row belongs to a different order.

#### Reviews per order
**Question:** can one order have more than one review, and how different are they?

In [4]:
per_order = reviews.groupby("order_id").size()
dist = per_order.value_counts().sort_index().rename("n_orders").to_frame()
dist.index.name = "reviews_per_order"
dist["pct_orders"] = (dist["n_orders"] / len(per_order) * 100).round(2)
display(dist)
print("orders with more than 1 review:", (per_order > 1).sum(), "| rows in them:", per_order[per_order > 1].sum())

,n_orders,pct_orders
reviews_per_order,,
1,98126,99.45
2,543,0.55
3,4,0.00


orders with more than 1 review: 547 | rows in them: 1098


- 98,126 orders have one review, 543 have two and 4 have three. In total 547 orders (1,098 rows) have more than one review.

#### Missing values
**Question:** which columns have gaps, and how do the two comment columns relate?

In [5]:
# Column-wise missing count and percentage, for every column (zeros included)
missing = pd.DataFrame({"n_rows": len(reviews),
                        "n_missing": reviews.isna().sum(),
                        "pct_missing": (reviews.isna().mean() * 100).round(2)})
display(missing.sort_values("n_missing", ascending=False))

# Comment text with blank (spaces only) messages treated as no message
reviews["message"] = reviews["review_comment_message"].str.strip().replace("", np.nan)
reviews["has_message"] = reviews["message"].notna()
reviews["has_title"] = reviews["review_comment_title"].notna()

print("blank messages (spaces only)        :", reviews["review_comment_message"].notna().sum() - reviews["has_message"].sum())
display(pd.crosstab(reviews["has_title"], reviews["has_message"]).rename(index={False: "no title", True: "title"},
                                                                         columns={False: "no message", True: "message"}))

,n_rows,n_missing,pct_missing
review_comment_title,99224,87656,88.34
review_comment_message,99224,58247,58.70
review_id,99224,0,0.00
review_score,99224,0,0.00
order_id,99224,0,0.00
review_creation_date,99224,0,0.00
review_answer_timestamp,99224,0,0.00


blank messages (spaces only)        : 27


has_message,no message,message
has_title,,
no title,56536,31120
title,1738,9830


- Only the two comment columns have gaps: title is missing in 87,656 rows (88.34%) and message in 58,247 rows (58.70%). All other columns are complete.
- 27 messages contain only spaces. They count as present in the missing table but carry no text, so they are treated as no message.
- Most reviews have no text at all: 56,536 have neither title nor message. 31,120 have a message only, 9,830 have both and 1,738 have a title only.
- A missing comment means the customer chose not to write.

#### Review score
**Question:** how are the scores distributed, and what share is "bad" (1 or 2)?

In [6]:
score = reviews["review_score"].value_counts().sort_index().rename("n_reviews").to_frame()
score.index.name = "review_score"
score["pct"] = (score["n_reviews"] / len(reviews) * 100).round(2)
display(score)

print("scores outside 1 to 5:", (~reviews["review_score"].between(1, 5)).sum())
print("bad reviews (score 1 or 2), share of rows:", round((reviews["review_score"] <= 2).mean() * 100, 2), "%")

,n_reviews,pct
review_score,,
1,11424,11.51
2,3151,3.18
3,8179,8.24
4,19142,19.29
5,57328,57.78


scores outside 1 to 5: 0
bad reviews (score 1 or 2), share of rows: 14.69 %


- 5 stars: 57.8% of rows, 4 stars: 19.3%, 3 stars: 8.2%, 1 star: 11.5%, 2 stars: 3.2%.
- No score is outside 1 to 5.
- The scores are U-shaped. People mostly rate 5 or 1, and the middle is thin.

#### Comments vs score
**Question:** is a missing comment random, or does it depend on the score?

In [7]:
by_score = reviews.groupby("review_score").agg(
    n_reviews=("review_id", "size"),
    pct_with_message=("has_message", "mean"),
    pct_with_title=("has_title", "mean"),
    median_message_length=("message", lambda s: s.dropna().str.len().median()),
)
by_score["pct_with_message"] = (by_score["pct_with_message"] * 100).round(1)
by_score["pct_with_title"] = (by_score["pct_with_title"] * 100).round(1)
display(by_score)

bad = reviews["review_score"] <= 2
print("bad reviews with a written message  :", f"{reviews.loc[bad, 'has_message'].mean() * 100:.1f}%")
print("bad share among reviews WITH message   :", f"{(reviews.loc[reviews['has_message'], 'review_score'] <= 2).mean() * 100:.1f}%")
print("bad share among reviews WITHOUT message:", f"{(reviews.loc[~reviews['has_message'], 'review_score'] <= 2).mean() * 100:.1f}%")

,n_reviews,pct_with_message,pct_with_title,median_message_length
review_score,,,,
1,11424,76.5,16.4,89.0
2,3151,68.1,15.2,87.0
3,8179,43.5,10.1,70.0
4,19142,31.2,9.1,46.0
5,57328,35.8,11.6,42.0


bad reviews with a written message  : 74.7%
bad share among reviews WITH message   : 26.6%
bad share among reviews WITHOUT message: 6.3%


- The comment is clearly not missing at random. 76.5% of 1-star reviews have a message, against 35.8% of 5-star reviews. The lowest rate is at 4 stars (31.2%).
- Unhappy customers also write more: the median message is 89 characters at 1 star and 42 at 5 stars.
- About three in four bad reviews carry text (the share is printed above), so text covers most of the complaints.

#### Comment text
**Question:** what does the text look like (length, format, repeated phrases, spelling)?

In [8]:
msg = reviews["message"].dropna()
length = msg.str.len()

display(length.describe(percentiles=[0.05, 0.5, 0.95, 0.99]).round(1).to_frame("message length (characters)"))
print("messages with 3 characters or fewer:", (length <= 3).sum())
print("messages with a line break         :", msg.str.contains(r"[\r\n]").sum())
print("messages with a web link           :", msg.str.contains("http|www", case=False).sum())
print("messages with non-ASCII characters :", f"{msg.map(lambda s: not s.isascii()).mean() * 100:.1f}%", "(accents)")

# Most frequent whole messages, lower case, with an English gloss
GLOSS = {"muito bom": "very good", "bom": "good", "recomendo": "recommend it", "otimo": "great (no accent)",
         "ótimo": "great (accent)", "excelente": "excellent", "ok": "ok", "tudo ok": "all ok", "boa": "good (feminine)",
         "ótimo produto": "great product"}
top = msg.str.strip().str.lower().value_counts().head(10).rename("n_reviews").to_frame()
top["english_gloss"] = top.index.map(GLOSS)
display(top)

same_text = msg.duplicated(keep=False)
print("messages sharing their exact text with another review:", same_text.sum(), "| distinct repeated texts:", msg[same_text].nunique())

,message length (characters)
count,40950.0
mean,68.4
std,53.8
min,1.0
5%,8.0
50%,53.0
95%,192.0
99%,200.0
max,208.0


messages with 3 characters or fewer: 809
messages with a line break         : 2710
messages with a web link           : 6
messages with non-ASCII characters : 55.3% (accents)


,n_reviews,english_gloss
message,,
muito bom,472,very good
bom,342,good
recomendo,234,recommend it
otimo,227,great (no accent)
ótimo,197,great (accent)
excelente,158,excellent
ok,149,ok
tudo ok,93,all ok
boa,91,good (feminine)


messages sharing their exact text with another review: 6505 | distinct repeated texts: 1171


- Messages are short: median 53 characters, 95th percentile 192, and nothing longer than 208. The 99th percentile is 200, so there seems to be a cap near 200 characters.
- 809 messages have 3 characters or fewer, 2,710 contain a line break, and only 6 contain a web link.
- 55.3% of messages use non-ASCII characters (accents). The text is real Portuguese in UTF-8, unlike the city names in the customers table, which had no accents.
- The same word appears with and without an accent: `otimo` (227) and `ótimo` (197) count as different strings.
- The most frequent whole messages are all short praise: `muito bom` (very good, 472), `bom` (good, 342), `recomendo` (recommend it, 234).

In [9]:
def strip_accents(s):
    return "".join(ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch)).lower()

text = reviews.loc[reviews["has_message"], ["review_score", "message"]].copy()
text["t"] = text["message"].map(strip_accents)
text["group"] = pd.cut(text["review_score"], [0, 2, 3, 5], labels=["bad (1-2)", "neutral (3)", "good (4-5)"])

patterns = {"nao recebi (did not receive)": "nao recebi", "atras (late, delay)": "atras", "defeito (defect)": "defeito",
            "errad (wrong)": "errad", "qualidade (quality)": "qualidade", "recomend (recommend)": "recomend"}

print("comments per group:", text["group"].value_counts().to_dict())
rows = []
for label, pat in patterns.items():
    has = text["t"].str.contains(pat)
    row = {"pattern": label, "n_comments": int(has.sum())}
    for grp in ["bad (1-2)", "neutral (3)", "good (4-5)"]:
        row[grp + " %"] = round(has[text["group"] == grp].mean() * 100, 1)
    rows.append(row)
pd.DataFrame(rows).set_index("pattern")

comments per group: {'good (4-5)': 26505, 'bad (1-2)': 10889, 'neutral (3)': 3556}


,n_comments,bad (1-2) %,neutral (3) %,good (4-5) %
pattern,,,,
nao recebi (did not receive),2079,16.7,5.0,0.3
"atras (late, delay)",509,2.8,1.9,0.5
defeito (defect),431,2.5,1.5,0.4
errad (wrong),503,3.3,1.9,0.3
qualidade (quality),2749,3.9,4.8,8.1
recomend (recommend),4639,3.9,4.0,15.4


- `nao recebi` (did not receive) is the clearest complaint: 16.7% of the comments on bad reviews contain it, against 0.3% of good ones (2,079 comments).
- `atras` (late, delay) is in 2.8% of bad-review comments, `defeito` (defect) in 2.5% and `errad` (wrong) in 3.3%. These are small, so the text has many different complaints, not a single dominant one.
- `recomend` (recommend) shows the reverse: 15.4% of good comments and 3.9% of bad ones.
- The text separates good from bad. A proper taxonomy is built from the data in Part 4.

#### Response time
**Question:** how long after the creation date does the customer answer, and does it depend on the score?

In [10]:
lag_days = (reviews["review_answer_timestamp"] - reviews["review_creation_date"]).dt.total_seconds() / 86400

display(lag_days.describe(percentiles=[0.01, 0.5, 0.95, 0.99]).round(2).to_frame("days from creation to answer"))
print("answers before creation:", (lag_days < 0).sum())
print("answers after more than 30 days :", (lag_days > 30).sum())
print("answers after more than 100 days:", (lag_days > 100).sum())

print("\nmedian days by score")
lag_days.groupby(reviews["review_score"]).median().round(2).to_frame("median_days")

,days from creation to answer
count,99224.00
mean,3.15
std,9.89
min,0.09
1%,0.38
50%,1.67
95%,6.97
99%,22.07
max,518.70


answers before creation: 0
answers after more than 30 days : 682
answers after more than 100 days: 172

median days by score


,median_days
review_score,
1,1.55
2,1.70
3,1.76
4,1.79
5,1.66


- The median answer comes 1.67 days after the creation date, the 95th percentile 6.97 days and the 99th percentile 22.07 days. No answer comes before the creation date.
- A long tail exists: 682 answers come after more than 30 days, 172 after more than 100 days, and the maximum is 518.7 days.
- The median is almost the same for every score (1.55 to 1.79 days), so answer speed does not separate happy from unhappy customers.

#### Outliers
**Question:** are there extreme or impossible values?

In [11]:
print("scores outside 1 to 5                  :", (~reviews["review_score"].between(1, 5)).sum())
print("answers before creation date           :", (lag_days < 0).sum())
print("answers after more than 100 days       :", (lag_days > 100).sum(), "| maximum (days):", round(lag_days.max(), 1))
print("longest message (characters)           :", length.max())
print("creation dates with a time of day      :", (~at_midnight).sum())

print("\nanswers with the longest delay")
reviews.assign(lag_days=lag_days.round(1)).nlargest(3, "lag_days")[["review_id", "review_score", "review_creation_date", "review_answer_timestamp", "lag_days"]]

scores outside 1 to 5                  : 0
answers before creation date           : 0
answers after more than 100 days       : 172 | maximum (days): 518.7
longest message (characters)           : 208
creation dates with a time of day      : 85

answers with the longest delay


,review_id,review_score,review_creation_date,review_answer_timestamp,lag_days
41620,40dad6438b6cbec46d936bec2377778c,1,2017-03-24,2018-08-24 16:46:52,518.7
66672,eeb94b15f672afaf38e8d74b6ddb25f8,5,2017-02-14,2018-07-11 15:47:37,512.7
22059,083e71bf744a3219a07a948f12243f9a,5,2017-03-28,2018-08-18 01:36:09,508.1


- Scores are always 1 to 5, no answer comes before its creation date, and the longest message is 208 characters. These parts are clean.
- The only extreme values are late answers: 172 after more than 100 days, up to 518.7 days. The answer date looks valid, so the customer simply answered very late.
- 85 creation dates have a time of day while the rest are midnight. This is a format difference, not an error.